# Pyramidal HEALPix convolution on real Sentinel-2 data (level 17)

This notebook applies `HealPixKernelPyramid`/`HealPixPyramidConv`
(see `docs/pyramid_convolution.md`) to a real Sentinel-2 L2A acquisition,
read directly from the public GRID4EARTH bucket at HEALPix level 17
(the coarsest published level, so the cheapest to read), using the same
data source as `dino_tuning_single_date.ipynb`
(`g4e_source.py`, next to this notebook).

**What this notebook demonstrates**: NaN-aware multiscale filtering (the
clouds and tile-edge gaps in Sentinel-2 are real holes, not synthetic ones)
on real data, applied to all 3 RGB bands *at once* through a single
multi-channel kernel pyramid (`channels=3`, block-diagonal across channels
-- no cross-band mixing), and the effect of the pyramid's depth (`Jmax`) on
how well those holes get filled -- a single, isolated `HealPixConv` kernel
(one band, no pyramid) can only fill a hole smaller than its own compact
footprint; the coarser pyramid levels see further.

**Important -- not executed here.** This notebook was written and reviewed
carefully, but **not run**: the cloud environment used to develop and test
`HealPixKernelPyramid`/`HealPixPyramidConv` (see
`compte_rendu_convolution_pyramidale.md`) has no network access to
`data.grid4earth.eu` (blocked by that sandbox's network policy). Run it in
an environment that has access to that bucket and to the
`xarray`/`zarr`/`obstore`/`cartopy`/`healpix_plot`/`umap` packages already
used by the existing DINO notebooks (e.g. Datarmor, or any machine where
`dino_tuning_single_date.ipynb` already works). If a cell fails, that is
not a known-and-fixed regression -- it is this notebook's first real run.


## 1. Parameters

In [6]:
import os, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))          # to import g4e_source (next to this notebook)
sys.path.insert(0, str(Path.cwd().parent))    # to import healpix_analyse in dev mode

from g4e_source import G4E_L2A, G4E_PRODUCTS, RGB, ProductSeries

from healpix_analyse.decomp import HealPixDecomp
from healpix_analyse.kernel_pyramid import HealPixKernelPyramid, kernel_gaussian
from healpix_analyse.pyramid_conv import HealPixPyramidConv
from healpix_analyse.dino import nested_to_tiles

# --------------------------------------------------------------------------
# TO SET
# --------------------------------------------------------------------------
PRODUCTS    = list(G4E_PRODUCTS)     # same products as dino_tuning_single_date.ipynb
TIME_INDEX  = 0                      # which product of PRODUCTS to read
DATA_LEVEL  = 17                     # coarsest published HEALPix level (17, 19, or 20)
SCALING     = "reflectance"          # "reflectance" (/10000), "percentile", "none"
CACHE       = os.path.expanduser("~/s2_cache")   # None to disable disk caching

N_CHANNELS         = 3    # R, G, B -- filtered together through one block-diagonal kernel pyramid
JMAX               = 4    # number of Down stages in the pyramid (see §7 for the effect of this choice)
COMPACT_KERNEL_SZ  = 5    # (odd) size of the per-band HealPixConv kernel
SIGMA_PIX          = 1.2  # width of the Gaussian kernel, in pixels of EACH band (docs/pyramid_convolution.md §A.2)
GAUGE_TYPE         = "phi"  # simple gauge; a Sentinel-2 scene is far from the geographic poles

DTYPE  = torch.float32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


device: cuda


## 2. Reading a real acquisition (GRID4EARTH, level 17)

Same as §2 of `dino_tuning_single_date.ipynb`: each product is a separate
zarr store, the HEALPix level is a **group** in it
(`measurements/reflectance/<level>`), and the first read downloads the
product (subsequent reads hit the disk cache if `CACHE` is set).


In [9]:
src = ProductSeries(PRODUCTS, level=DATA_LEVEL, base=G4E_L2A, bands=RGB, scaling=SCALING)
level, cell_id, dates = src.level, src.cell_id, src.dates

print(f"HEALPix level {level}, {cell_id.size} cells, product {dates[TIME_INDEX]}")
print("ellipsoid declared by the store:", src.ellipsoid)

t0 = time.time()
rgb = src.rgb(TIME_INDEX, cache=CACHE)     # [N, 3] reflectances in [0, 1]; NaN = missing/masked
print(f"read in {time.time() - t0:.0f}s; NaN: {100 * np.isnan(rgb).any(1).mean():.1f}% of cells")
src.ellipsoid='WGS84'

2 products, level 17, 69632 cells, bands ['b01', 'b02', 'b03', 'b04', 'b05', 'b06', 'b07', 'b09', 'b11', 'b12', 'b8a'], ellipsoid wgs84
HEALPix level 17, 69632 cells, product 2025-05-22
ellipsoid declared by the store: wgs84
read in 0s; NaN: 0.0% of cells


## 3. Pyramid and kernel pyramid, matched to the store's real geometry

Two points worth noting (see `docs/pyramid_convolution.md`):

- **§A.5, ellipsoid.** This module defaults to `ellipsoid="sphere"`, but
  EOPF products generally declare `"wgs84"`. We pass `ellipsoid=src.ellipsoid`
  explicitly to both `HealPixDecomp` and `HealPixKernelPyramid.from_kernel`
  so both use the same geometry as the store's real cells -- otherwise
  distances would be silently mislabeled. (The ellipsoid name is resolved
  case-insensitively against `healpix_geo`'s own registry, so the store's
  lowercase `"wgs84"` and the module's own `"WGS84"` default both work.)
- **Multi-channel kernel.** `channels=N_CHANNELS` builds one
  `in_channels=out_channels=3` `HealPixConv` per band, with a kernel that is
  block-diagonal across channels: R, G and B are filtered with the
  *identical* profile, independently, with **no** cross-channel mixing. This
  lets the three bands be convolved through the pyramid in a single call,
  each keeping its own NaN/cloud mask.


In [10]:
decomp = HealPixDecomp(
    level=level, cell_ids=cell_id, Jmax=JMAX,
    ellipsoid=src.ellipsoid, dtype=DTYPE, device=DEVICE,
)
print(decomp)
print("band sizes (fine -> coarse):", decomp.sizes)

t0 = time.time()
kernel_pyramid = HealPixKernelPyramid.from_kernel(
    decomp, kernel_gaussian(sigma_pix=SIGMA_PIX),
    compact_kernel_sz=COMPACT_KERNEL_SZ, gauge_type=GAUGE_TYPE,
    channels=N_CHANNELS, ellipsoid=src.ellipsoid, dtype=DTYPE,
)
print(f"kernel pyramid built in {time.time() - t0:.0f}s ({decomp.n_bands} bands, {N_CHANNELS} channels each)")

pconv = HealPixPyramidConv(decomp, kernel_pyramid, mode="normalized")


HealPixDecomp(
  level=17, Jmax=4, scales=4, bands=5, domain=partial, sizes=(69632, 17408, 4352, 1088, 272), filter='symmetric Gaussian', up_norm='col_l1'
  (down_layers): ModuleList(
    (0-3): 4 x HealPixDown()
  )
  (up_layers): ModuleList(
    (0-3): 4 x HealPixUp()
  )
)
band sizes (fine -> coarse): (69632, 17408, 4352, 1088, 272)


TypeError: HealPixKernelPyramid.from_kernel() got an unexpected keyword argument 'channels'

## 4. Masked pyramidal convolution on the 3 bands

In [11]:
# [N, 3] -> [3, N]: with channels=N_CHANNELS the kernel pyramid expects
# (and correctly returns) [channels, N] -- see docs/pyramid_convolution.md §C
# and the class docstring of HealPixKernelPyramid for why this differs from
# plugging a [C, N] array into a channels=1 kernel pyramid (that silently
# comes back [C, 1, N], not [C, N]).
x = torch.as_tensor(rgb.T, dtype=DTYPE, device=DEVICE)

t0 = time.time()
with torch.no_grad():
    y, support = pconv(x, return_support=True)
print(f"pyramidal convolution (3 bands) in {time.time() - t0:.0f}s, output shape {tuple(y.shape)}")

y_np = y.detach().cpu().numpy() if torch.is_tensor(y) else np.asarray(y)
support_np = support.detach().cpu().numpy() if torch.is_tensor(support) else np.asarray(support)
rgb_filtered = y_np.T                 # [N, 3]
support_map = support_np[0]           # [N] -- identical across the 3 bands (same starting mask shape)

nan_before = 100 * np.isnan(rgb).any(1).mean()
nan_after = 100 * np.isnan(rgb_filtered).any(1).mean()
print(f"NaN before filtering                 : {nan_before:.1f}% of cells")
print(f"NaN after (pyramid, Jmax={JMAX})        : {nan_after:.1f}% of cells")
print(f"synthesized support: min {np.nanmin(support_map):.3g}, "
      f"max {np.nanmax(support_map):.3g}, "
      f"median (where finite) {np.nanmedian(support_map):.3g}")


pyramidal convolution (3 bands) in 0s, output shape (3, 1, 69632)
NaN before filtering                 : 0.0% of cells
NaN after (pyramid, Jmax=4)        : 62.5% of cells
synthesized support: min 0, max 8.48, median (where finite) 0


## 5. Visual comparison: tiles before / after

Same tiling as the DINO notebooks (`nested_to_tiles`), to look closely at
what the filter actually did on real data (not just global statistics).


In [12]:
def to_rgb(tile, p_low=2, p_high=98):
    """[3, S, S] -> displayable image, percentile-stretched (NaN in black)."""
    img = np.transpose(tile, (1, 2, 0))
    finite = img[np.isfinite(img)]
    if finite.size == 0:
        return np.zeros_like(img)
    lo, hi = np.nanpercentile(finite, [p_low, p_high])
    return np.clip((np.nan_to_num(img, nan=lo) - lo) / max(hi - lo, 1e-6), 0, 1)


TILE_LEVELS = 8   # tiles of 2**TILE_LEVELS px; increase if the thumbnails below are too small
parent_level = level - TILE_LEVELS
if parent_level < 0:
    raise ValueError(f"TILE_LEVELS={TILE_LEVELS} > level={level}; reduce TILE_LEVELS")

tiles_before, parent_ids, coverage, _ = nested_to_tiles(rgb, cell_id, level, parent_level, fill="nan")
tiles_after, parent_ids2, coverage2, _ = nested_to_tiles(rgb_filtered, cell_id, level, parent_level, fill="nan")
assert np.array_equal(parent_ids, parent_ids2), "both tilings must cover the same tiles"

order = np.argsort(-coverage)[:6]   # best-covered tiles first (the most readable)
fig, axes = plt.subplots(2, len(order), figsize=(2.4 * len(order), 5.2), squeeze=False)
for j, i in enumerate(order):
    axes[0, j].imshow(to_rgb(tiles_before[i]))
    axes[0, j].set_title(f"before  cov={coverage[i]:.2f}", fontsize=8)
    axes[1, j].imshow(to_rgb(tiles_after[i]))
    axes[1, j].set_title("after (pyramid)", fontsize=8)
    for a in axes[:, j]:
        a.set_axis_off()
fig.tight_layout()
plt.show()


ValueError: data must have shape [N, C] or [N], got (69632, 1, 3)

## 6. Full HEALPix map

In [ ]:
import cartopy.crs as ccrs
import healpix_plot

grid = healpix_plot.HealpixGrid(level=level, indexing_scheme="nested", ellipsoid=src.ellipsoid)

fig, axes = plt.subplots(1, 3, figsize=(20, 6),
                          subplot_kw={"projection": ccrs.PlateCarree()}, layout="constrained")
hi = float(np.nanpercentile(rgb, 98))
healpix_plot.plot(cell_id, rgb, healpix_grid=grid, sampling_grid={"shape": 900},
                   ax=axes[0], rgb_clip=(0.0, hi), axis_labels="none",
                   title=f"{dates[TIME_INDEX]}  raw RGB (level {level})")
healpix_plot.plot(cell_id, rgb_filtered, healpix_grid=grid, sampling_grid={"shape": 900},
                   ax=axes[1], rgb_clip=(0.0, hi), axis_labels="none",
                   title=f"after pyramidal convolution (Jmax={JMAX}, {COMPACT_KERNEL_SZ}x{COMPACT_KERNEL_SZ})")
mp = healpix_plot.plot(cell_id, support_map, healpix_grid=grid, sampling_grid={"shape": 900},
                        ax=axes[2], axis_labels="none",
                        title="synthesized support (confidence after filtering)")
fig.colorbar(mp, ax=axes[2], shrink=0.7)
plt.show()


## 7. Effect of pyramid depth (`Jmax`) on hole-filling

An isolated `HealPixConv` kernel (a single band, which is what `Jmax=0`
amounts to) can only fill a hole smaller than its own compact footprint
(`COMPACT_KERNEL_SZ`). Coarser pyramid levels (larger `Jmax`) see a wider
spatial context at each additional stage, at the cost of one more kernel to
build. Here we measure, on this acquisition's real holes (clouds, tile
edges), the fraction of cells that get any non-zero support after
synthesis -- without looking at the values, just "is there a response at
all".


In [ ]:
JMAX_VALUES = sorted(set([0, 1, 2, JMAX]))
x1 = x[:1]   # one channel is enough: all 3 bands share the same mask shape

coverage_by_jmax = {}
for jm in JMAX_VALUES:
    d = HealPixDecomp(level=level, cell_ids=cell_id, Jmax=jm,
                       ellipsoid=src.ellipsoid, dtype=DTYPE, device=DEVICE)
    kp = HealPixKernelPyramid.from_kernel(
        d, kernel_gaussian(SIGMA_PIX), compact_kernel_sz=COMPACT_KERNEL_SZ,
        gauge_type=GAUGE_TYPE, channels=1, ellipsoid=src.ellipsoid, dtype=DTYPE,
    )
    pc = HealPixPyramidConv(d, kp, mode="normalized")
    with torch.no_grad():
        _, supp = pc(x1, return_support=True)
    supp_np = supp.detach().cpu().numpy() if torch.is_tensor(supp) else np.asarray(supp)
    frac_supported = float((supp_np[0] > 1e-6).mean())
    coverage_by_jmax[jm] = frac_supported
    print(f"Jmax={jm:>2d} ({d.n_bands} band(s)): {100 * frac_supported:.1f}% of cells have support > 0")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(list(coverage_by_jmax.keys()), [100 * v for v in coverage_by_jmax.values()], "o-")
ax.set_xlabel("Jmax (pyramid stages)")
ax.set_ylabel("% cells with support > 0")
ax.set_title("Real hole-filling vs. pyramid depth")
fig.tight_layout()
plt.show()


## Final notes

- This notebook deliberately reuses `g4e_source.py` as-is (same reading,
  same caching, same normalization) to stay consistent with the existing
  DINO notebooks -- no data is reprojected or resampled beyond what
  `HealPixDecomp`/`HealPixConv` already do.
- The `channels=N_CHANNELS` kernel pyramid (§3-4) filters R, G, B in a
  single call with a block-diagonal, no-cross-mixing kernel -- see the
  `HealPixKernelPyramid` class docstring, and
  `tests/test_kernel_pyramid.py`/`tests/test_pyramid_conv.py` for the
  regression tests that pin this shape and independence down numerically.
- The `ellipsoid=src.ellipsoid` choice (§3) matters and is documented as an
  open limitation in `docs/pyramid_convolution.md` (§A.5, §E point 5):
  nothing automatically checks that a `HealPixDecomp`/`HealPixKernelPyramid`
  built with different ellipsoids are not accidentally used together.
- `JMAX`, `COMPACT_KERNEL_SZ`, `SIGMA_PIX` and `GAUGE_TYPE` are deliberately
  grouped at the top of the notebook (§1) so they can be replayed without
  re-reading everything; the network read (§2) is the most expensive part,
  hence the disk cache.
- No comparison here against the independent oracle in
  `healpix_analyse.validation` (already done on synthetic data in
  `tests/test_kernel_pyramid.py` and documented in
  `docs/pyramid_convolution.md` §E): this notebook is a demonstration on
  real data, not a new numerical validation campaign.
